# 03 — Baseline Model: TF-IDF + Logistic Regression

Fast, CPU-only baseline. Loads the processed dataset from notebook 02,
trains, evaluates, and saves the model to `models/`.

In [ ]:
%pip install pandas scikit-learn matplotlib seaborn joblib python-dotenv --quiet

In [ ]:
import sys, os
from pathlib import Path

# Make the project root importable so `from src... import ...` works
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

print("Project root:", PROJECT_ROOT)

In [ ]:
import pandas as pd
from src.config import PROCESSED_DATA_DIR

df = pd.read_csv(PROCESSED_DATA_DIR / "dataset.csv")
print(f"Loaded {len(df)} rows")
df.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"], df["sentiment"],
    test_size=0.2, random_state=42, stratify=df["sentiment"]
)

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words="english")
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

baseline_clf = LogisticRegression(max_iter=1000, class_weight="balanced")
baseline_clf.fit(X_train_vec, y_train)
print("Trained.")

In [ ]:
from src.evaluate import print_report, plot_confusion_matrix
from sklearn.metrics import accuracy_score, f1_score

y_pred_baseline = baseline_clf.predict(X_test_vec)
print_report(y_test, y_pred_baseline, model_name="TF-IDF + Logistic Regression")
plot_confusion_matrix(y_test, y_pred_baseline, model_name="Baseline", cmap="Blues")

baseline_accuracy = accuracy_score(y_test, y_pred_baseline)
baseline_f1 = f1_score(y_test, y_pred_baseline, average="macro")
print(f"\nAccuracy: {baseline_accuracy:.3f} | Macro F1: {baseline_f1:.3f}")

In [ ]:
import joblib
from src.config import MODELS_DIR

joblib.dump(baseline_clf, MODELS_DIR / "baseline_model.joblib")
joblib.dump(vectorizer, MODELS_DIR / "baseline_vectorizer.joblib")

# Save metrics so notebook 05 can pull them in for the comparison
import json
with open(MODELS_DIR / "baseline_metrics.json", "w") as f:
    json.dump({"accuracy": baseline_accuracy, "f1_macro": baseline_f1}, f)

print("Saved model, vectorizer, and metrics.")

Next: open **04_transformer_model.ipynb** to train DistilBERT.